<a href="https://colab.research.google.com/github/BruCarnauba/miniguia-estudos-notebooklm/blob/main/Assistente_de_Voz_Multi_Idiomas_Com_Whisper_e_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
language = 'pt'

# 1. Capturando dúvida do usuário por áudio 🎤

In [12]:
# @title
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=10):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Me diga em poucas palavras qual é a sua dúvida. Estou ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Me diga em poucas palavras qual é a sua dúvida. Estou ouvindo...



<IPython.core.display.Javascript object>

In [19]:
record_file = '/content/request_audio.wav'

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [14]:
# @title
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
# @title
import whisper

# Assuming 'pt' (Portuguese) is the desired language for transcription
# This line is added to ensure the 'language' variable is defined
language = 'pt'

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("small")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 testando a gravação de áudio.


# 3. Integração com a API do Gemini 💬

In [27]:
# @title
# from openai import OpenAI
# import os

# # Configura a chave de API da OpenAI usando a variável de ambiente 'OPENAI_API_KEY'
# client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

# # Envia uma requisição à API do ChatCompletion usando o modelo GPT-3.5 Turbo
# # Lembrando que, a variável 'transcription' contém a transcrição do nosso áudio.
# response = client.chat.completions.create(
#     model="gpt-3.5-turbo",
#     messages=[ { "role": "user", "content": transcription } ]
# )

# # Obtém a resposta gerada pelo ChatGPT
# chatgpt_response = response.choices[0].message.content
# print(chatgpt_response)

## Usando a API Gemini do Google (Gratuita para Testes)

Para usar a API Gemini, você precisará de uma chave de API. Se ainda não tiver uma, crie uma no [Google AI Studio](https://makersuite.google.com/app/apikey).

No Colab, adicione a chave ao gerenciador de segredos em '🔑' no painel esquerdo. Dê a ela o nome `GOOGLE_API_KEY`. Em seguida, passe a chave para o SDK:

In [24]:
# @title
# Importa o SDK Python
import google.generativeai as genai
# Usado para armazenar chaves de API com segurança
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

Antes de fazer qualquer chamada à API, você precisa inicializar o modelo generativo.

In [22]:
# @title
# Inicializa a API Gemini
gemini_model = genai.GenerativeModel('gemini-pro')

Agora você pode fazer chamadas à API. Por exemplo, para obter uma resposta para a sua dúvida anterior:

In [26]:
# @title
import google.generativeai as genai

# Usando a transcrição anterior como prompt

# Diagnóstico: Listar modelos disponíveis para verificar o nome correto.
# O erro indica que 'gemini-pro' pode não ser o nome correto ou não suporta generateContent.
available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
print(f"Modelos disponíveis para generateContent: {available_models}")

# Tentar usar um modelo alternativo que suporte generateContent.
# 'gemini-pro' ou 'gemini-pro-vision' são os mais comuns para texto/imagem.
model_to_use = None
if 'gemini-pro' in available_models:
    model_to_use = 'gemini-pro'
elif 'gemini-pro-vision' in available_models:
    model_to_use = 'gemini-pro-vision'
elif available_models:
    # Se 'gemini-pro' e 'gemini-pro-vision' não estiverem disponíveis, tente o primeiro da lista
    model_to_use = available_models[0]

if model_to_use:
    print(f"Usando o modelo: {model_to_use}")
    # Re-initialize the model with the correct name.
    # A melhor prática seria corrigir a inicialização em cell 34e5aaa8.
    gemini_model = genai.GenerativeModel(model_to_use)
    response = gemini_model.generate_content(transcription)
    print(response.text)
else:
    print("Nenhum modelo Gemini adequado encontrado que suporte 'generateContent'. Por favor, verifique sua configuração da API ou disponibilidade de modelos para sua conta/região.")


Modelos disponíveis para generateContent: ['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-lite-001', 'models/gemini-2.0-flash-lite', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-3-1b-it', 'models/gemma-3-4b-it', 'models/gemma-3-12b-it', 'models/gemma-3-27b-it', 'models/gemma-3n-e4b-it', 'models/gemma-3n-e2b-it', 'models/gemma-4-26b-a4b-it', 'models/gemma-4-31b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-pro-preview', 'models/gemini-3-flash-preview', 'models/gemini-3.1-pro-preview', 'models/gemini-3.1-pro-preview-customtools', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3-pro-image-preview', 'models/nano-banana-pro-preview', 'models/gemini-3.1-flash-image-preview', 'models/lyria-3-clip-preview', 'models/lyria-3-

# 4. Sintetizando a Resposta do Gemini Como Voz (gTTS) 🔊

In [30]:
!pip install gTTS

In [31]:
!pip install gTTS
from gtts import  gTTS

# Cria um objeto gTTS com a resposta gerada pelo Gemini e a língua que será sintetizada em voz (variável "language").
# Usamos 'response.text' que vem da chamada à API Gemini.
gtts_object = gTTS(text=response.text, lang=language, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))